## はじめに

このノートブックでは、Part 1 で作成した Bronze 層のテキストを Cortex AI 関数で加工し、分析に使う **Silver 層**と、検索にそのまま使う **Gold 層**のテーブルを作成します。通話や PDF の中にしかなかった情報を列として取り出し、注文データと組み合わせて集計できることを体験します。

**前提条件:**
- `part1_20261216.ipynb` が完了していること（`RAW_CALL_TRANSCRIPTS`、`RAW_DOCUMENTS` などが作成済み）

**主な処理内容:**
1. AI 関数を試す（AI_COMPLETE, AI_REDACT）
2. Silver 層のテーブルを作成する（通話の分析、PDF の属性抽出）
3. 取り出した列を使って集計する
4. Gold 層の検索用チャンクを作成する（Part 3 の Cortex Search で使用）

**生成されるテーブル:**
- `CALL_ANALYSIS`（Silver）: 通話ごとの問い合わせ種別・商品カテゴリ・感情・解約リスク・要約と、個人情報をマスクした本文（98件）
- `DOC_ATTRIBUTES`（Silver）: PDF ごとのタイトル・文書の種類・部署・公開範囲・発行日・対象商品（11件）
- `KNOWLEDGE_CHUNKS`（Gold）: 通話と PDF を検索用に分割し、属性を付けたチャンク（200件）

> **料金について**: AI 関数は処理量に応じてクレジットを消費します。実行前に[最新の料金](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-cost)を確認してください。

> **途中で遅れた場合**: `checkpoint_20261216.sql` の「Part 1」で開始時点、「Part 2」で完了時点のテーブルを復元できます。

In [ ]:
-- ============================================================================
-- 環境設定
-- ============================================================================
-- 使用するロール、ウェアハウス、スキーマを設定
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE GLACIERSTYLE_WH;
USE SCHEMA GLACIERSTYLE_DB.EC_ANALYTICS_SCHEMA;

## 1. Cortex AI 関数を試してみよう

テーブル全体を処理する前に、数件だけで AI 関数の動きを確かめます。

### 1-1. AI_COMPLETE — 大規模言語モデルへの指示

`AI_COMPLETE` は、プロンプト（指示文）を大規模言語モデルに渡して、答えを受け取る関数です。通話の本文と指示を渡すだけで、問い合わせ種別を答えてもらえます。

`response_format` に JSON スキーマを書くと、答えがその形で返ります（構造化出力）。`enum` に選択肢を書いておくと、選択肢以外の値は返らないので、そのまま集計に使えます。


In [ ]:
-- ============================================================================
-- AI_COMPLETE: 通話から問い合わせ種別を取り出してみよう
-- ============================================================================
--   inquiry_free  : 指示だけ            → AI が自由に表現する（「返品」「配送状況の確認」など）
--   inquiry_choice: JSON スキーマで選択肢を指定 → 指定した中から選ばれ、集計しやすくなる
SELECT
    call_id,
    LEFT(transcript_text, 60) AS transcript_preview,
    TRIM(AI_COMPLETE(
        model => 'openai-gpt-5.4-mini',
        prompt => 'この通話の問い合わせ種別を、短い言葉で1つだけ答えてください。\n\n' || transcript_text,
        model_parameters => {'temperature': 0}
    )::VARCHAR) AS inquiry_free,
    AI_COMPLETE(
        model => 'openai-gpt-5.4-mini',
        prompt => 'この通話の問い合わせ種別を JSON で答えてください。\n\n' || transcript_text,
        model_parameters => {'temperature': 0},
        response_format => {
            'type': 'json',
            'schema': {
                'type': 'object',
                'properties': {
                    'inquiry_type': {'type': 'string',
                        'enum': ['商品不具合', '配送遅延', '返品・交換', '注文キャンセル', '商品問い合わせ', '支払い', '会員・ポイント', 'その他']}
                },
                'additionalProperties': False,
                'required': ['inquiry_type']
            }
        }
    ):inquiry_type::VARCHAR AS inquiry_choice
FROM RAW_CALL_TRANSCRIPTS
ORDER BY call_id
LIMIT 5;


### 1-2. AI_REDACT — 個人情報のマスク

通話には氏名や電話番号などの個人情報が含まれます。`AI_REDACT` で `[NAME]` のような記号に置き換えてから、検索や共有に使います。

In [ ]:
-- ============================================================================
-- AI_REDACT: 通話の個人情報をマスクしてみよう
-- ============================================================================
SELECT
    call_id,
    LEFT(transcript_text, 120) AS original_text,
    LEFT(AI_REDACT(transcript_text)::VARCHAR, 120) AS masked_text
FROM RAW_CALL_TRANSCRIPTS
ORDER BY call_id
LIMIT 3;

## 2. Silver 層のテーブルの作成

セクション1で試した AI 関数の考え方を全件に適用し、Silver 層のテーブルを作成します。

### 2-1. 通話の分析（CALL_ANALYSIS）

通話（`RAW_CALL_TRANSCRIPTS`、98件）から、`AI_COMPLETE` で問い合わせ種別・商品名・商品カテゴリ・感情・解約リスク・要約の6項目を取り出します。あわせて `AI_REDACT` で本文の個人情報をマスクします。

In [ ]:
-- ============================================================================
-- 通話の AI 分析と Silver 層への保存
-- ============================================================================
CREATE OR REPLACE TABLE CALL_ANALYSIS AS
WITH extracted AS (
    SELECT
        t.*,
        AI_COMPLETE(
            model => 'openai-gpt-5.4-mini',
            prompt => 'コールセンターの通話内容を分析し、JSON で回答してください。要約は必ず60文字以内にしてください。\n\n' || transcript_text,
            model_parameters => {'temperature': 0},
            response_format => {
                'type': 'json',
                'schema': {
                    'type': 'object',
                    'properties': {
                        'inquiry_type': {'type': 'string', 'description': '問い合わせ種別',
                            'enum': ['商品不具合', '配送遅延', '返品・交換', '注文キャンセル', '商品問い合わせ', '支払い', '会員・ポイント', 'その他']},
                        'product_name': {'type': 'string', 'description': '話題になっている商品名。なければ空文字'},
                        'product_category': {'type': 'string', 'description': '話題になっている商品のカテゴリ。商品の話でなければ「なし」',
                            'enum': ['照明', '家具', '収納', '調理器具', '食器', '文具', 'トラベル', 'ギフト', 'ファッション小物', 'ルームウェア', 'なし']},
                        'sentiment': {'type': 'string', 'description': '顧客の感情',
                            'enum': ['ネガティブ', '中立', 'ポジティブ']},
                        'churn_risk': {'type': 'string', 'description': '顧客が他社に乗り換えたり利用をやめたりする可能性',
                            'enum': ['高', '中', '低']},
                        'summary': {'type': 'string', 'description': '通話内容の要約（60文字以内）'}
                    },
                    'additionalProperties': False,
                    'required': ['inquiry_type', 'product_name', 'product_category', 'sentiment', 'churn_risk', 'summary']
                }
            }
        ) AS r
    FROM RAW_CALL_TRANSCRIPTS t
)
SELECT
    call_id, call_datetime, customer_id, order_id, product_id,
    r:inquiry_type::VARCHAR AS inquiry_type,
    NULLIF(r:product_name::VARCHAR, '') AS product_name,
    NULLIF(r:product_category::VARCHAR, 'なし') AS product_category,
    r:sentiment::VARCHAR AS sentiment,
    r:churn_risk::VARCHAR AS churn_risk,
    -- 指示しても長くなることがあるので、60文字で切りそろえる
    LEFT(r:summary::VARCHAR, 60) AS summary,
    -- AI_REDACT の結果は VARIANT で返るため、文字列に変換して保存
    AI_REDACT(transcript_text)::VARCHAR AS transcript_masked
FROM extracted;

-- 問い合わせ種別ごとの件数を確認
SELECT inquiry_type, COUNT(*) AS call_count
FROM CALL_ANALYSIS
GROUP BY inquiry_type
ORDER BY call_count DESC;


### 2-2. PDF の属性抽出（DOC_ATTRIBUTES）

PDF の1ページ目（表紙・文書情報）から、タイトル・文書の種類・部署・公開範囲・発行日・対象商品を取り出します。これらは Part 3 の Cortex Search で「品質管理部の文書だけ」「社外公開の文書だけ」のように検索を絞り込む属性になります。

2-1 と同じく `AI_COMPLETE` の構造化出力を使い、文書の種類・部署・公開範囲・商品カテゴリは `enum` で選択肢を固定します。商品IDは文字列の配列（`ARRAY`）として受け取ります。

店舗報告の売上表のように、文書の中に商品カテゴリ名が出てくると、AI がそれを対象の商品カテゴリと取り違えることがあります。商品IDが取れた文書だけに商品カテゴリを付けます。

In [ ]:
-- ============================================================================
-- PDF の1ページ目から属性を取り出し、Silver 層へ保存
-- ============================================================================
CREATE OR REPLACE TABLE DOC_ATTRIBUTES AS
WITH extracted AS (
    SELECT
        file_name,
        AI_COMPLETE(
            model => 'openai-gpt-5.4-mini',
            prompt => '社内文書の1ページ目（表紙・文書情報）です。文書の属性を JSON で答えてください。'
                || '公開範囲は、お客様向けの文書（FAQ など）は社外公開、社内の業務用の文書は社内限定とします。\n\n' || page_text,
            model_parameters => {'temperature': 0},
            response_format => {
                'type': 'json',
                'schema': {
                    'type': 'object',
                    'properties': {
                        'title': {'type': 'string', 'description': '文書のタイトル'},
                        'doc_type': {'type': 'string', 'description': '文書の種類',
                            'enum': ['FAQ', 'マニュアル', '品質報告', '通達', '店舗報告']},
                        'department': {'type': 'string', 'description': '作成した部署',
                            'enum': ['カスタマーサポート部', '品質管理部', '店舗運営部']},
                        'audience': {'type': 'string', 'description': '公開範囲。お客様向けの文書は社外公開',
                            'enum': ['社外公開', '社内限定']},
                        'doc_date': {'type': 'string', 'description': '発行日または最終更新日（YYYY-MM-DD）'},
                        'product_category': {'type': 'string', 'description': '対象の商品カテゴリ。特定の商品についての文書でなければ「なし」',
                            'enum': ['照明', '家具', '収納', '調理器具', '食器', '文具', 'トラベル', 'ギフト', 'ファッション小物', 'ルームウェア', 'なし']},
                        'product_ids': {'type': 'array', 'description': '対象の商品ID（例: PRD-LGT-0001）。なければ空の配列',
                            'items': {'type': 'string'}}
                    },
                    'additionalProperties': False,
                    'required': ['title', 'doc_type', 'department', 'audience', 'doc_date', 'product_category', 'product_ids']
                }
            }
        ) AS r
    FROM RAW_DOCUMENTS
    WHERE page_no = 1
)
SELECT
    file_name,
    r:title::VARCHAR AS title,
    r:doc_type::VARCHAR AS doc_type,
    r:department::VARCHAR AS department,
    r:audience::VARCHAR AS audience,
    TRY_TO_DATE(r:doc_date::VARCHAR) AS doc_date,
    -- 商品IDが取れた文書だけに商品カテゴリを付ける
    IFF(ARRAY_SIZE(r:product_ids::ARRAY) > 0, NULLIF(r:product_category::VARCHAR, 'なし'), NULL) AS product_category,
    r:product_ids::ARRAY AS product_ids
FROM extracted;

-- 抽出結果を確認
SELECT file_name, title, doc_type, department, audience, doc_date, product_category, product_ids
FROM DOC_ATTRIBUTES
ORDER BY file_name;


## 3. 取り出した列で集計してみよう

通話の中にしかなかった「問い合わせ種別」「感情」「解約リスク」が列になったので、通常のデータと同じように集計できます。さらに `order_id` で注文データとつなぐと、通話と注文の両方から同じ問題が見えてきます。

### 3-1. 月別の問い合わせ件数

In [ ]:
-- ============================================================================
-- 月別の通話件数
-- ============================================================================
SELECT
    TO_CHAR(DATE_TRUNC('month', call_datetime), 'YYYY-MM') AS call_month,
    COUNT(*) AS call_count,
    COUNT_IF(inquiry_type = '商品不具合') AS defect_calls,
    COUNT_IF(inquiry_type = '配送遅延') AS delay_calls,
    COUNT_IF(sentiment = 'ネガティブ') AS negative_calls
FROM CALL_ANALYSIS
GROUP BY call_month
ORDER BY call_month;

10月から「商品不具合」の通話が急に増えています。どの商品の、どのロットで起きているのかを、注文データとつないで確かめます。

### 3-2. 不具合の通話を注文データとつなぐ

In [ ]:
-- ============================================================================
-- 「商品不具合」の通話を、商品と入荷ロット別に集計
-- ============================================================================
SELECT
    p.product_name,
    COALESCE(o.lot_no, '（ロット記録なし）') AS lot_no,
    COUNT(*) AS defect_calls,
    COUNT_IF(a.churn_risk = '高') AS high_churn_risk_calls
FROM CALL_ANALYSIS a
JOIN FACT_ORDERS o ON o.order_id = a.order_id
JOIN DIM_PRODUCTS p ON p.product_id = o.product_id
WHERE a.inquiry_type = '商品不具合'
GROUP BY p.product_name, lot_no
ORDER BY defect_calls DESC;

In [ ]:
-- ============================================================================
-- 注文データだけで見た、LEDデスクライト 調光式の月別返品率
-- ============================================================================
SELECT
    TO_CHAR(DATE_TRUNC('month', o.order_datetime), 'YYYY-MM') AS order_month,
    COALESCE(o.lot_no, '（ロット記録なし）') AS lot_no,
    COUNT(*) AS order_count,
    ROUND(COUNT_IF(o.order_status = 'returned') / COUNT(*) * 100, 1) AS return_rate_pct
FROM FACT_ORDERS o
WHERE o.product_id = 'PRD-LGT-0001'
GROUP BY order_month, lot_no
ORDER BY order_month, lot_no;

通話から取り出した「商品不具合」はロット `LOT-2609-B` に集中しており、注文データでもこのロットの返品率が10月以降に大きく上がっています。通話という非構造化データが、注文データと同じように分析に使えるようになりました。

原因や対応方針は PDF の品質報告書や通達に書かれています。Part 3 以降では、これらの文書を検索できるようにし、AI エージェントが数値と文書の両方から回答できるようにします。

## 4. 検索用チャンクの作成

Cortex Search で検索する単位（チャンク）を作ります。通話は1件を1チャンクにし、マスク済みの本文の先頭に要約を付けます。PDF はページ本文を約800文字ずつに分割します。どちらにも、セクション2で取り出した属性（文書の種類・部署・公開範囲・日付・商品カテゴリなど）を付けます。

In [ ]:
-- ============================================================================
-- 通話と PDF を統合した検索用チャンクテーブル
-- ============================================================================
-- 通話の種類・部署・公開範囲は、コールセンターの通話という出どころで決まる
CREATE OR REPLACE TABLE KNOWLEDGE_CHUNKS AS
-- 通話（1通話 = 1チャンク）
SELECT
    call_id AS chunk_id,
    '【要約】' || summary || '\n' || transcript_masked AS chunk_text,
    inquiry_type || COALESCE(' / ' || product_name, '') AS title,
    '通話' AS doc_type,
    'カスタマーサポート部' AS department,
    '社内限定' AS audience,
    call_datetime::DATE AS doc_date,
    product_category,
    IFF(product_id IS NULL, ARRAY_CONSTRUCT(), ARRAY_CONSTRUCT(product_id)) AS product_ids,
    sentiment
FROM CALL_ANALYSIS
UNION ALL
-- PDF（ページ本文を分割し、文書の属性を付ける）
SELECT
    d.file_name || '-p' || d.page_no || '-' || c.index AS chunk_id,
    c.value::VARCHAR AS chunk_text,
    a.title, a.doc_type, a.department, a.audience, a.doc_date,
    a.product_category, a.product_ids,
    NULL AS sentiment
FROM RAW_DOCUMENTS d
JOIN DOC_ATTRIBUTES a ON a.file_name = d.file_name,
LATERAL FLATTEN(input => SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(d.page_text, 'markdown', 800, 100)) c;

-- 文書の種類と公開範囲ごとのチャンク数を確認
SELECT doc_type, audience, COUNT(*) AS chunk_count
FROM KNOWLEDGE_CHUNKS
GROUP BY doc_type, audience
ORDER BY doc_type;

## 5. データ確認

作成した Silver 層と Gold 層のテーブルの件数を確認します。

In [ ]:
-- ============================================================================
-- Silver 層と Gold 層のテーブルのレコード数を確認
-- ============================================================================
SELECT 'CALL_ANALYSIS' AS table_name, COUNT(*) AS record_count, '通話の分析結果（98件）' AS description FROM CALL_ANALYSIS
UNION ALL
SELECT 'DOC_ATTRIBUTES', COUNT(*), 'PDF の属性（11件）' FROM DOC_ATTRIBUTES
UNION ALL
SELECT 'KNOWLEDGE_CHUNKS', COUNT(*), '検索用チャンク（200件）' FROM KNOWLEDGE_CHUNKS;

## まとめ

このノートブックでは、Bronze 層のテキストを Cortex AI 関数で加工し、Silver 層と Gold 層のテーブルを作成しました。

### 作成したテーブル一覧
- `CALL_ANALYSIS`（Silver）: 通話の問い合わせ種別・商品カテゴリ・感情・解約リスク・要約と、マスク済みの本文
- `DOC_ATTRIBUTES`（Silver）: PDF のタイトル・文書の種類・部署・公開範囲・発行日・対象商品
- `KNOWLEDGE_CHUNKS`（Gold）: 属性付きの検索用チャンク

### 使用した Cortex AI 関数
- `AI_COMPLETE`（構造化出力）: 通話の6項目と PDF の文書の属性を、JSON スキーマで取り出す（選択肢を指定すると結果がそろう）
- `AI_REDACT`: 個人情報のマスク
- `SPLIT_TEXT_RECURSIVE_CHARACTER`: 検索用のチャンク分割

### わかったこと
- 10月以降、「商品不具合」の通話が急増している
- 不具合の通話は LEDデスクライト 調光式のロット `LOT-2609-B` に集中し、注文データでも同じロットの返品率が上がっている

### 次のステップ
- **Part 3**: `KNOWLEDGE_CHUNKS` から Cortex Search サービスを作成し、属性で絞り込んで検索します

## 補足：その他の Cortex AI 関数

> **このパートは時間が余った方向けです。** 本編のテーブルからは参照しないので、飛ばしても Part 3 以降でエラーにはなりません。
>
> AI 関数を追加で実行するため、AI クレジットを消費します。全部を実行すると約 0.7 クレジットかかります（検証時の実績）。

補足パートでは、本編で使わなかった AI 関数を試します。作成するテーブルには `SUP_` を付けて、本編のテーブルと区別します。

| 節 | 使うデータ | 使う AI 関数 |
| --- | --- | --- |
| A. SNS の分析 | SNS 投稿（`sns_logs.json`） | AI_SENTIMENT, AI_CLASSIFY, AI_FILTER, AI_SIMILARITY |
| B. 広告クリエイティブの分析 | 広告の画像とコピー（`ad_creatives.csv`） | AI_COMPLETE（画像）, AI_CLASSIFY（画像） |
| C. 通話の集約・絞り込み・翻訳 | 本編の `CALL_ANALYSIS` | AI_AGG, AI_FILTER, AI_TRANSLATE |
| D. PDF の表の取り出し | 店舗の月次報告 PDF | AI_EXTRACT（表） |
| E. PDF の画像の取り出し | 運営マニュアル PDF | AI_PARSE_DOCUMENT（画像） |
| F. AI_EXTRACT で通話から取り出す | 本編の `RAW_CALL_TRANSCRIPTS` | AI_EXTRACT |

> **料金について**: AI 関数は処理量に応じてクレジットを消費します。実行前に[最新の料金](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-cost)を確認してください。

### A. SNS の分析

SNS の投稿（265件、2026年10月15日〜11月30日）をテーブルにロードし、感情と投稿の種類を判定します。

#### A-1. SNS 投稿のロード

JSON ファイルは、ステージから直接 `SELECT` してテーブルにできます。

In [ ]:
-- ============================================================================
-- SNS 投稿（JSON）を RAW_SNS_MENTIONS にロード
-- ============================================================================
-- JSON の配列を1件ずつの行として読むファイル形式
CREATE OR REPLACE FILE FORMAT SUP_JSON_FORMAT TYPE = JSON STRIP_OUTER_ARRAY = TRUE;

CREATE OR REPLACE TABLE RAW_SNS_MENTIONS AS
SELECT
    $1:post_id::VARCHAR AS post_id,
    $1:platform::VARCHAR AS platform,
    $1:username::VARCHAR AS username,
    $1:content::VARCHAR AS content,
    $1:posted_at::TIMESTAMP_TZ AS posted_at,
    $1:likes::NUMBER AS likes,
    $1:replies::NUMBER AS replies,
    $1:hashtags AS hashtags
FROM @DATA_STAGE/sns_logs.json (FILE_FORMAT => 'SUP_JSON_FORMAT');

-- 件数と期間を確認（265件）
SELECT COUNT(*) AS post_count, MIN(posted_at) AS first_post, MAX(posted_at) AS last_post
FROM RAW_SNS_MENTIONS;

#### A-2. AI_SENTIMENT — 感情分析

テキストが positive / negative / neutral / mixed のどれかを判定します。

In [ ]:
-- ============================================================================
-- AI_SENTIMENT: SNS 投稿の感情を判定してみよう
-- ============================================================================
SELECT
    content,
    AI_SENTIMENT(content):categories[0]:sentiment::VARCHAR AS sentiment
FROM RAW_SNS_MENTIONS
LIMIT 5;

#### A-3. AI_CLASSIFY — テキスト分類

テキストを、指定したラベルのどれかに分類します。ラベルは自由に決められます。

**チャレンジ**: 下のセルのラベル `['称賛', 'クレーム', '質問', '提案']` を、自分で考えたラベルに書き換えて実行してみましょう（例: `'購入報告'`、`'比較検討'`）。

In [ ]:
-- ============================================================================
-- AI_CLASSIFY: SNS 投稿を分類してみよう
-- ============================================================================
-- ラベルを書き換えて、分類結果がどう変わるか試してください
SELECT
    content,
    AI_CLASSIFY(content, ['称賛', 'クレーム', '質問', '提案']):labels[0]::VARCHAR AS post_type
FROM RAW_SNS_MENTIONS
LIMIT 5;

#### A-4. SNS の分析テーブルの作成

感情・投稿の種類・商品名・商品カテゴリを全件に付けて、`SUP_SNS_ANALYZED` を作成します。商品名とカテゴリは `AI_EXTRACT` で取り出します。

> **ヒント**: 265件を処理するため、実行に1〜2分かかります。

In [ ]:
-- ============================================================================
-- SNS 投稿の AI 分析
-- ============================================================================
CREATE OR REPLACE TABLE SUP_SNS_ANALYZED AS
SELECT
    post_id,
    platform,
    posted_at,
    likes,
    content,
    AI_SENTIMENT(content):categories[0]:sentiment::VARCHAR AS sentiment,
    AI_CLASSIFY(content, ['称賛', 'クレーム', '質問', '提案']):labels[0]::VARCHAR AS post_type,
    AI_EXTRACT(content, {
        'product_name': '投稿で言及されている商品名',
        'category': '商品カテゴリを次から1つ: インテリア, キッチン, ライフスタイル雑貨, アパレル'
    }):response AS extracted,
    -- 商品名が書かれていない投稿は 'None' という文字列が返るので空にする
    NULLIF(extracted:product_name::VARCHAR, 'None') AS product_name,
    NULLIF(extracted:category::VARCHAR, 'None') AS category
FROM RAW_SNS_MENTIONS;

-- 感情と投稿の種類の組み合わせごとの件数
SELECT sentiment, post_type, COUNT(*) AS post_count
FROM SUP_SNS_ANALYZED
GROUP BY ALL
ORDER BY post_count DESC;

#### A-5. SNS にも出ていた不具合の声

本編では、通話と注文のデータから LEDデスクライト 調光式の不具合を見つけました。SNS でも、10月以降に同じ症状の投稿が出ています。

「ちらつ」のようなキーワードで探すと、「急に暗くなったり明るくなったりする」のような言い方を見落とします。ここでは `AI_FILTER` で、自然文で書いた条件に合う投稿を探します。`AI_FILTER` は TRUE / FALSE を返すので、`WHERE` 句や `COUNT_IF` で使えます（C-2 でも使います）。

> **ヒント**: 条件はできるだけ具体的に書きます。「不具合を訴えているか」だけだと、「商品に傷があった」のような別の不具合の投稿も含まれます。

In [ ]:
-- ============================================================================
-- AI_FILTER: デスクライトの「ちらつき」投稿を探す
-- ============================================================================
CREATE OR REPLACE TEMP TABLE SUP_SNS_FLICKER AS
SELECT
    posted_at,
    sentiment,
    content,
    -- キーワードで探した場合
    content ILIKE '%ちらつ%' OR content ILIKE '%チカチカ%' AS keyword_match,
    -- AI_FILTER で探した場合
    AI_FILTER(PROMPT(
        'この投稿で、投稿者は自分が買った照明について、ちらつく・明るさが勝手に変わるという症状を報告していますか？ {0}',
        content
    )) AS ai_match
FROM SUP_SNS_ANALYZED
WHERE product_name ILIKE '%デスクライト%';

-- 月別の件数（キーワードでは6件、AI_FILTER では8件）
SELECT
    DATE_TRUNC('month', posted_at)::DATE AS post_month,
    COUNT(*) AS desk_light_posts,
    COUNT_IF(keyword_match) AS keyword_count,
    COUNT_IF(ai_match) AS ai_filter_count
FROM SUP_SNS_FLICKER
GROUP BY 1
ORDER BY 1;

In [ ]:
-- ============================================================================
-- AI_FILTER で見つかった投稿の本文（keyword_match が FALSE の投稿に注目）
-- ============================================================================
SELECT posted_at, sentiment, keyword_match, content
FROM SUP_SNS_FLICKER
WHERE ai_match
ORDER BY posted_at;

#### A-6. AI_SIMILARITY — 商品マスタとの名寄せ

SNS の商品名は、略称や言い換えで書かれることがあります。`AI_SIMILARITY` で商品マスタ（`DIM_PRODUCTS`）の商品名との近さを 0〜1 で計算し、いちばん近い商品に紐付けます。

全商品と比べると呼び出し回数が増えるので、A-4 で取り出したカテゴリが同じ商品だけを候補にします。

In [ ]:
-- ============================================================================
-- SNS の商品名と商品マスタの名寄せ
-- ============================================================================
CREATE OR REPLACE TABLE SUP_SNS_PRODUCT_MATCH AS
SELECT
    s.post_id,
    s.product_name AS mentioned_name,
    p.product_id,
    p.product_name AS master_name,
    AI_SIMILARITY(s.product_name, p.product_name) AS similarity
FROM SUP_SNS_ANALYZED s
JOIN DIM_PRODUCTS p ON p.category_l1 = s.category
WHERE s.product_name IS NOT NULL
-- 投稿ごとに、いちばん近い商品だけを残す
QUALIFY ROW_NUMBER() OVER (PARTITION BY s.post_id ORDER BY similarity DESC) = 1;

-- 類似度の低いものから確認する
SELECT mentioned_name, master_name, ROUND(similarity, 2) AS similarity, COUNT(*) AS post_count
FROM SUP_SNS_PRODUCT_MATCH
GROUP BY ALL
ORDER BY similarity
LIMIT 10;

類似度が低いものは、正しく紐付いていない可能性があります。たとえば「クリップ式LEDライト」が文具の商品に紐付いているのは、A-4 でカテゴリの判定を誤ったためです。実際の業務では、しきい値（例: 0.7）未満を人が確認する、という運用と組み合わせます。

### B. 広告クリエイティブの分析

広告15本の画像とコピーを分析し、クリック率（CTR）・コンバージョン率（CVR）と並べます。

#### B-1. 広告データのロード

In [ ]:
-- ============================================================================
-- 広告クリエイティブ（CSV）を RAW_AD_CREATIVES にロード
-- ============================================================================
CREATE OR REPLACE TABLE RAW_AD_CREATIVES (
    creative_id VARCHAR,
    campaign_id VARCHAR,
    creative_name VARCHAR,
    creative_type VARCHAR,
    image_file_path VARCHAR,
    copy_text VARCHAR,
    headline VARCHAR,
    cta_text VARCHAR,
    target_segment VARCHAR,
    platform VARCHAR,
    start_date DATE,
    end_date DATE,
    impressions NUMBER,
    clicks NUMBER,
    conversions NUMBER,
    spend NUMBER(12, 2)
);

COPY INTO RAW_AD_CREATIVES
FROM @DATA_STAGE/ad_creatives.csv
FILE_FORMAT = (TYPE = CSV SKIP_HEADER = 1 FIELD_OPTIONALLY_ENCLOSED_BY = '"');

SELECT creative_id, creative_name, image_file_path, headline FROM RAW_AD_CREATIVES;

#### B-2. AI_COMPLETE — 画像の分析

`AI_COMPLETE` に `TO_FILE` で画像を渡すと、画像の内容について質問できます。ここでは色・構図・印象を JSON で答えてもらいます。あわせて `AI_CLASSIFY` に画像を渡して、画像のスタイルを分類します。

> **ヒント**: 画像を扱えるモデルはリージョンによって異なります。このノートブックでは `openai-gpt-5.4-mini` を使います。

In [ ]:
-- ============================================================================
-- AI_COMPLETE: 広告画像を分析してみよう
-- ============================================================================
SELECT
    image_file_path,
    AI_COMPLETE(
        'openai-gpt-5.4-mini',
        'この広告画像の主な色・構図・全体の印象を、{"main_colors": ["色1", "色2"], "composition": "構図", "impression": "印象"} の JSON で答えてください。JSON のみを出力してください。',
        TO_FILE('@DATA_STAGE', 'ad_images/' || image_file_path)
    ) AS visual_analysis
FROM RAW_AD_CREATIVES
LIMIT 3;

#### B-3. 広告の分析テーブルの作成

画像の分析に加えて、コピー（見出し・本文・ボタンの文言）から訴求の種類を `AI_EXTRACT` で取り出し、成果の指標と並べて `SUP_AD_CREATIVE_ANALYSIS` を作成します。

In [ ]:
-- ============================================================================
-- 広告クリエイティブの AI 分析
-- ============================================================================
CREATE OR REPLACE TABLE SUP_AD_CREATIVE_ANALYSIS AS
SELECT
    c.creative_id,
    c.creative_name,
    c.creative_type,
    c.platform,
    c.target_segment,
    c.headline,
    c.copy_text,
    c.impressions,
    c.clicks,
    c.conversions,
    c.spend,
    ROUND(c.clicks / NULLIF(c.impressions, 0) * 100, 2) AS ctr_pct,
    ROUND(c.conversions / NULLIF(c.clicks, 0) * 100, 2) AS cvr_pct,
    -- 画像の色・構図・印象
    TRY_PARSE_JSON(AI_COMPLETE(
        'openai-gpt-5.4-mini',
        'この広告画像の主な色・構図・全体の印象を、{"main_colors": ["色1", "色2"], "composition": "構図", "impression": "印象"} の JSON で答えてください。JSON のみを出力してください。',
        TO_FILE('@DATA_STAGE', 'ad_images/' || c.image_file_path)
    )) AS visual,
    -- 画像のスタイル
    AI_CLASSIFY(
        TO_FILE('@DATA_STAGE', 'ad_images/' || c.image_file_path),
        ['商品フォーカス', 'ライフスタイル', 'テキスト中心', 'キャンペーン訴求']
    ):labels[0]::VARCHAR AS image_style,
    -- コピーの訴求の種類
    AI_EXTRACT(c.headline || ' ' || c.copy_text || ' ' || c.cta_text, {
        'appeal_type': '訴求の種類を次から1つ: 品質訴求, 価格訴求, 限定訴求, 感情訴求'
    }):response:appeal_type::VARCHAR AS appeal_type
FROM RAW_AD_CREATIVES c;

-- 訴求の種類ごとの平均 CTR・CVR
SELECT
    appeal_type,
    COUNT(*) AS creative_count,
    ROUND(AVG(ctr_pct), 2) AS avg_ctr_pct,
    ROUND(AVG(cvr_pct), 2) AS avg_cvr_pct
FROM SUP_AD_CREATIVE_ANALYSIS
GROUP BY 1
ORDER BY avg_ctr_pct DESC;

In [ ]:
-- ============================================================================
-- 広告ごとの分析結果を確認
-- ============================================================================
SELECT
    creative_name,
    image_style,
    appeal_type,
    visual:main_colors AS main_colors,
    visual:impression::VARCHAR AS impression,
    ctr_pct,
    cvr_pct
FROM SUP_AD_CREATIVE_ANALYSIS
ORDER BY ctr_pct DESC;

### C. 通話の集約・絞り込み・翻訳

本編で作成した `CALL_ANALYSIS` を使って、複数行をまとめる・条件で絞り込む・翻訳する AI 関数を試します。

#### C-1. AI_AGG — 複数の通話をまとめて要約

`AI_AGG` は `SUM` や `COUNT` と同じ集計関数です。グループ内の全行を読んで、指示どおりにまとめます。「商品不具合」の通話28件（ほとんどが LOT-2609-B のデスクライト）を1つの要約にします。

In [ ]:
-- ============================================================================
-- AI_AGG: 不具合の通話をまとめて要約
-- ============================================================================
SELECT
    COUNT(*) AS call_count,
    AI_AGG(
        summary,
        '商品の不具合に関するお客様の声を、症状・お客様の要望・当社の対応の3点で、300文字以内の日本語でまとめてください。'
    ) AS voice_summary
FROM CALL_ANALYSIS
WHERE inquiry_type = '商品不具合';

#### C-2. AI_FILTER — 条件に合う行の絞り込み

A-5 で使った `AI_FILTER` を通話にも使います。「返送料を当社が負担すると案内したか」で絞り込むと、特例対応の対象である不具合の通話だけが残ります。

In [ ]:
-- ============================================================================
-- AI_FILTER: 返送料を当社負担と案内した通話を数える
-- ============================================================================
SELECT
    inquiry_type,
    COUNT(*) AS call_count,
    COUNT_IF(AI_FILTER(PROMPT(
        'この通話で、オペレーターは返送料を当社（店側）が負担すると案内していますか？ {0}',
        transcript_masked
    ))) AS matched_count
FROM CALL_ANALYSIS
GROUP BY 1
ORDER BY matched_count DESC;

#### C-3. AI_TRANSLATE — 翻訳

海外の仕入先に報告する場面を想定して、通話の要約を英語に翻訳します。

In [ ]:
-- ============================================================================
-- AI_TRANSLATE: 不具合の通話の要約を英訳
-- ============================================================================
SELECT
    call_id,
    summary,
    AI_TRANSLATE(summary, 'ja', 'en') AS summary_en
FROM CALL_ANALYSIS
WHERE inquiry_type = '商品不具合'
ORDER BY call_datetime DESC
LIMIT 3;

### D. PDF の表の取り出し

実店舗の売上は、店舗運営部が毎月作る PDF の報告書にしか載っていません。`AI_EXTRACT` で表を取り出してテーブルにすると、EC の売上と並べて集計できます。

表を取り出すときは、`responseFormat` に `schema` を書き、表（`type: object`）の列と並び順（`column_ordering`）を指定します。

In [ ]:
-- ============================================================================
-- 店舗の月次報告 PDF から、店舗別・カテゴリ別の売上表を取り出す
-- ============================================================================
CREATE OR REPLACE TABLE SUP_STORE_SALES AS
WITH extracted AS (
    SELECT
        relative_path,
        AI_EXTRACT(
            file => TO_FILE('@DATA_STAGE', relative_path),
            responseFormat => {
                'schema': {
                    'type': 'object',
                    'properties': {
                        'report_month': {'type': 'string', 'description': '報告の対象年月（YYYY-MM 形式）'},
                        'sales_table': {
                            'type': 'object',
                            'description': '店舗別・カテゴリ別売上の表',
                            'column_ordering': ['store', 'interior', 'kitchen', 'lifestyle', 'apparel', 'total'],
                            'properties': {
                                'store': {'type': 'array', 'description': '店舗'},
                                'interior': {'type': 'array', 'description': 'インテリア'},
                                'kitchen': {'type': 'array', 'description': 'キッチン'},
                                'lifestyle': {'type': 'array', 'description': 'ライフスタイル雑貨'},
                                'apparel': {'type': 'array', 'description': 'アパレル'},
                                'total': {'type': 'array', 'description': '合計'}
                            }
                        }
                    }
                }
            }
        ):response AS r
    FROM DIRECTORY(@DATA_STAGE)
    WHERE relative_path LIKE 'documents/store_report_%.pdf'
)
-- 表は列ごとの配列で返るので、店舗の配列を展開して同じ位置の値を取り出す
SELECT
    TO_DATE(r:report_month::VARCHAR || '-01') AS report_month,
    s.value::VARCHAR AS store_name,
    TO_NUMBER(REPLACE(r:sales_table:interior[s.index]::VARCHAR, ',', '')) AS interior_sales,
    TO_NUMBER(REPLACE(r:sales_table:kitchen[s.index]::VARCHAR, ',', '')) AS kitchen_sales,
    TO_NUMBER(REPLACE(r:sales_table:lifestyle[s.index]::VARCHAR, ',', '')) AS lifestyle_sales,
    TO_NUMBER(REPLACE(r:sales_table:apparel[s.index]::VARCHAR, ',', '')) AS apparel_sales,
    TO_NUMBER(REPLACE(r:sales_table:total[s.index]::VARCHAR, ',', '')) AS total_sales,
    relative_path AS source_file
FROM extracted,
LATERAL FLATTEN(input => r:sales_table:store) s;

-- 6店舗 × 6か月 = 36行。カテゴリの合計が「合計」列と一致するかも確認する
SELECT
    COUNT(*) AS row_count,
    COUNT_IF(interior_sales + kitchen_sales + lifestyle_sales + apparel_sales <> total_sales) AS mismatch_count
FROM SUP_STORE_SALES;

In [ ]:
-- ============================================================================
-- 店舗の売上と EC の売上を月別に並べる
-- ============================================================================
WITH store AS (
    SELECT report_month, SUM(total_sales) AS store_sales
    FROM SUP_STORE_SALES
    GROUP BY 1
),
ec AS (
    SELECT DATE_TRUNC('month', order_datetime)::DATE AS order_month, SUM(total_amount) AS ec_sales
    FROM FACT_ORDERS
    WHERE order_status <> 'cancelled'
    GROUP BY 1
)
SELECT
    store.report_month,
    store.store_sales,
    ec.ec_sales,
    ROUND(ec.ec_sales / (store.store_sales + ec.ec_sales) * 100, 1) AS ec_share_pct
FROM store
LEFT JOIN ec ON ec.order_month = store.report_month
ORDER BY 1;

11月は、店舗・EC ともにブラックフライデーセールで売上が伸びています。PDF にしかなかった数字も、テーブルにすれば注文データと同じように SQL で扱えます。

### E. PDF の画像の取り出し

`AI_PARSE_DOCUMENT` に `'extract_images': TRUE` を指定すると、PDF 内の画像を Base64 で取り出せます。取り出した画像は、ストアドプロシージャでファイルに戻してステージに保存します。

In [ ]:
-- ============================================================================
-- 取り出した画像をステージに保存するストアドプロシージャ
-- ============================================================================
CREATE OR REPLACE PROCEDURE SAVE_EXTRACTED_IMAGES(config OBJECT)
RETURNS TABLE(status STRING, message STRING)
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'run'
AS
$$
import base64
import io
import os


def iter_images(data):
    # 画像は結果の直下か、ページごとの images に入っている
    images = data.get("images") or [img for page in data.get("pages", []) for img in page.get("images", [])]
    for image in images:
        header, payload = image["image_base64"].split(",", 1)
        extension = header.split("/")[1].split(";")[0]
        yield image["id"], extension, payload


def run(session, config):
    destination = config["DESTINATION_PATH"]
    saved = 0
    for image_id, extension, payload in iter_images(config["PARSE_DOCUMENT_RESULT"]):
        name = os.path.splitext(image_id)[0] + "." + extension
        session.file.put_stream(io.BytesIO(base64.b64decode(payload)), f"{destination}/{name}",
                                auto_compress=False, overwrite=True)
        saved += 1
    return session.create_dataframe([["SUCCESS", f"{saved} 枚の画像を保存しました"]], schema=["status", "message"])
$$;

In [ ]:
-- ============================================================================
-- 運営マニュアル PDF から画像を取り出して保存
-- ============================================================================
CREATE STAGE IF NOT EXISTS EXTRACTED_IMAGES_STAGE DIRECTORY = (ENABLE = TRUE)
  ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE');

CALL SAVE_EXTRACTED_IMAGES(OBJECT_CONSTRUCT(
    'DESTINATION_PATH', '@EXTRACTED_IMAGES_STAGE',
    'PARSE_DOCUMENT_RESULT', (
        SELECT AI_PARSE_DOCUMENT(
            TO_FILE('@DATA_STAGE', 'operation_manual_w_images.pdf'),
            {'mode': 'LAYOUT', 'extract_images': TRUE}
        )
    )
));

-- 保存された画像を確認（マニュアルの画像は1枚）
LIST @EXTRACTED_IMAGES_STAGE;

保存した画像は、Snowsight の **カタログ » データベースエクスプローラー** で `EXTRACTED_IMAGES_STAGE` を開くと確認できます。


### F. AI_EXTRACT で通話から取り出す

本編 1-1 では `AI_COMPLETE` で問い合わせ種別を取り出しました。同じことは `AI_EXTRACT` でもできます。

テキストから、指定した項目を取り出します。取り出したい項目と説明を `responseFormat` に書くだけで、結果が JSON で返ります。

- **AI_EXTRACT**: プロンプトやモデルを書かずに、項目と説明だけで取り出せます。PDF や画像のファイルから直接取り出すこともできます（補足 D で使用）
- **AI_COMPLETE（構造化出力）**: モデルを選び、`enum` で選択肢を固定できます。決まった選択肢で多くの件数を処理するときに向いています（本編 2-1 で使用）


In [ ]:
-- ============================================================================
-- AI_EXTRACT: 通話から問い合わせ種別を取り出してみよう
-- ============================================================================
--   inquiry_free  : '問い合わせ種別' → AI が自由に表現する（「返品」「配送状況の確認」など）
--   inquiry_choice: 選択肢を書く     → 指定した中から選ばれ、集計しやすくなる
SELECT
    call_id,
    LEFT(transcript_text, 60) AS transcript_preview,
    AI_EXTRACT(
        text => transcript_text,
        responseFormat => {'inquiry_type': '問い合わせ種別'}
    ):response:inquiry_type::VARCHAR AS inquiry_free,
    AI_EXTRACT(
        text => transcript_text,
        responseFormat => {'inquiry_type': '問い合わせ種別を次から1つ: 商品不具合, 配送遅延, 返品・交換, 注文キャンセル, 商品問い合わせ, 支払い, 会員・ポイント, その他'}
    ):response:inquiry_type::VARCHAR AS inquiry_choice
FROM RAW_CALL_TRANSCRIPTS
ORDER BY call_id
LIMIT 5;

### 補足のまとめ

| テーブル | 内容 |
| --- | --- |
| `RAW_SNS_MENTIONS` / `SUP_SNS_ANALYZED` | SNS 投稿と、感情・投稿の種類・商品名 |
| `SUP_SNS_PRODUCT_MATCH` | SNS の商品名と商品マスタの名寄せ |
| `RAW_AD_CREATIVES` / `SUP_AD_CREATIVE_ANALYSIS` | 広告と、画像の印象・スタイル・訴求の種類・CTR/CVR |
| `SUP_STORE_SALES` | 店舗の月次報告 PDF から取り出した売上表 |

いずれも Part 3 以降では使いません。